In [0]:

# 1. Import PySpark functions
from pyspark.sql import functions as F

# 2. Read Bronze table
df = spark.table("workspace.retail_project.bronze_sales")

# 3. Convert date and monetary columns
df_clean = (
    df
    .withColumn(
        "ORDERDATE",
        F.to_date(
            F.col("ORDERDATE"),
            "M/d/yyyy H:mm"
        )
    )
    .withColumn(
        "SALES",
        F.col("SALES").cast("decimal(18,2)")
    )
    .withColumn(
        "PRICEEACH",
        F.col("PRICEEACH").cast("decimal(18,2)")
    )
    .withColumn(
        "MSRP",
        F.col("MSRP").cast("decimal(18,2)")
    )
)

# 4. Remove unnecessary optional column
df_clean = df_clean.drop("ADDRESSLINE2")

# 5. Check invalid dates
invalid_dates = df_clean.filter(
    F.col("ORDERDATE").isNull()
)

print("Invalid dates:", invalid_dates.count())

# 6. Check invalid sales records
invalid_sales = df_clean.filter(
    F.col("SALES").isNull() |
    (F.col("SALES") < 0) |
    F.col("PRICEEACH").isNull() |
    (F.col("PRICEEACH") < 0) |
    F.col("QUANTITYORDERED").isNull() |
    (F.col("QUANTITYORDERED") <= 0)
)

print("Invalid sales records:", invalid_sales.count())

# 7. Check duplicate order lines
duplicate_orders = (
    df_clean.groupBy(
        "ORDERNUMBER",
        "ORDERLINENUMBER"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate order lines:", duplicate_orders.count())

# 8. Stop if validation fails
if (
    invalid_dates.count() > 0
    or invalid_sales.count() > 0
    or duplicate_orders.count() > 0
):
    raise ValueError(
        "Data validation failed. Review invalid records."
    )

# 9. Save Silver Delta table
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.silver_sales")

# 10. Verify saved table
silver_df = spark.table(
    "workspace.retail_project.silver_sales"
)

print("Silver rows:", silver_df.count())
print("Silver columns:", len(silver_df.columns))

display(silver_df.limit(10))
